# Manual Preprocessing
## Imports

In [ ]:
import pandas as pd
import numpy as np

import os
import re
import zipfile
import pathlib
import random
import pprint
from pathlib import Path
from typing import List, Optional
from dataclasses import dataclass
from collections import Counter
from music21 import pitch

import matplotlib
# matplotlib.use("Agg")
import matplotlib.pyplot as plt

import seaborn as sns
sns.set_theme(
    style="whitegrid",
    context="talk",
)

## Kern syntax

In the Humdrum syntax, columns of data are called SPINES. SPINES of information may change
position within a file; they may disappear, or split into several columns, for example.

Based on the information found in the Github repo with the Essen files [cite], the music files contain
several indicators that need to be taken into account:

### INTERPRETATIONS

  Labels that indicate the type of information being represented in each Spine. They are
  represented as two asterisks (**). We also have *SPINE-PATH TERMINATORS*, they indicate the end
  of their respective spine, represented as *-

### TANDEM INTERPRETATIONS

  Tandem interpretations are identified by a single leading asterisk (*).
  Tandem interpretations encode additional or supplementary information, for example the names
  of the voices

### COMMENTS

In any representation, some information may best be conveyed as
  COMMENTS rather than as part of the encoded data. In Humdrum formats they are expressed as
  exclamation marks:
  - *GLOBAL COMMENTS*: they begin with two exclamation marks
  - *LOCAL COMMENTS*: they begin with one exclamation mark
  - *NULL LOCAL COMMENTS*: exclamation marks with no text

### PITCH

Enconded through a scheme of upper and lower case letters:
  - c      middle C (i.e. C4)
  - cc     C an octave higher than middle C (C5)
  - C      C an octave lower than middle C (C3)
  - CC     C two octaves lower than middle C (C2)
  - B      B below middle C (B3)
  - b      B a major seventh above middle C (B4)
  - d#     D-sharp above middle C (D#4)
  - d##    D double-sharp above middle C
  - d###   D tiple-sharp above middle C
  - e-     E-flat above middle C (enharmonically equivalent to d##)
  - BB--   B double-flat; augmented ninth below middle C
  - cn     C natural, middle C

  Sharps, flats, and naturals are mutually exclusive in **kern, so tokens such as "cc#n" and
  "GG-#" are illegal. Something to take into account when cleaning the data, in case some errors
  are present.

### SLURS, TIES, PHRASES

  - *Phrases*: indicated by {}
  - *Slurs*: indicated by () --> ligadura entre notas distintas
  - *Ties*: indicated by [] --> ligadura entre misma nota

  They can be nested and elided (the overlapping of phrases, where the last note or chord
  of one phrase serves simultaneously as the first note or chord of the next)

### ORNAMENTS

  - "T" and "t": whole-tone and semitone TRILLS
  - "M" and "m": whole-tone and semitone MORDENTS
  - "W" and "w": whole-tone and semitone INVERTED MORDENTS
  - "S":  a TURN
  - dollar sign ("$"): an inverted (or Wagnerian) TURN
  - "R":  When a concluding turn is appended to the end of an ornament, "R" is
    added to the ornament signifier (as in "tR" and "TR").
  - ":" (multi-note) ARPEGGIATION
  - "0": presence of ornaments OTHER than trills, mordents, inverted mordents, and turns.

### ARTICULATION MARKS

  - (') for STACCATO
  - double-quote (") for PIZZICATO
  - greve (`) for ATTACCA
  - tilde (~) for TENUTO
  - caret (^) for all note-related accents (including < and >)

### UP & DOWN BOWS

  - up-bow (v) and down-bow (u)

### STEMS (plica)

  - "/": Up-stem
  - "\": Down-stem

### DURATIONS

0       breve duration (redonda)
  1       whole duration
  2       half duration
  4       quarter duration
  8       eighth duration
  16      sixteenth duration
  32      thirty-second duration
  64      sixty-fourth duration

  *Dotted notes*:
    2.	dotted half duration
    8..	doubly-dotted eighth duration

  The semicolon (;) denotes a pause.

### GRACE NOTES & GROUPETTOS

  - *ACCIACCATURAS*: "q", e.g "G#q"
  - *GROUPETTOS*: "Q", e.g minature sixteenth-note middle C would be encoded as "16cQ"
  - *APPOGGIATURAS*: The appoggiatura note itself is designated by the upper-case letter "P", whereas
    the subsequent note (whose notated duration has been shorted) is designated by the
    lower-case letter "p".

### BEAMING

  - beginning and ends of beams are signified by the upper-case letters
    "L" and "J" respectively
  - Partial beams may extend to the right (K) or left (k)
  - each beam is designated by its own L-J pair, e.g:
    16..LL
    64JJkk
 
### BARLINES

  -  0-9	measure numbers
  -  a-z	alternate measures
  -  ;	        pause
  -  =	        barline
  -  ==	        double barline
  -  |	        normal width visual rendering
  -  !	        heavy width visual rendering
  -  '   	partial barline (from second to fourth line)
  -  `  	partial barline (short stroke at top of staff)
  -  -  	invisible barline
  -  :  	repeat sign

#### Examples
  -  =	        unnumbered barline
  -  =29 	the beginning of measure 29
  -  =29; 	the beginning of measure 29 with pause
  -  =29a	first occurrence of measure 29
  -  =29c	third occurrence of measure 29
  -  =29c;	third occurrence of measure 29 with pause
  -  ==	        double barline
  -  ==;	double barline with pause
  -  =|	        unnumbered barline, normal line width
  -  =!	        unnumbered barline, heavy line width
  -  ==|!	double barline, normal line followed by heavy line
  -  =29|	beginning of measure 29, normal line width
  -  =:|:	barline with left and right repeats, normal line width
  -  =:||:	barline with left and right repeats, two normal-width lines
  -  ='	        unnumbered barline, rendered with partial barline (mid)
  -  =29`	beginning of measure 29, rendered with partial barline (top)
  -  =29-	beginning of measure 29, no barline drawn
  -  ==:|!	double barline with repeat, normal/heavy lines
  -  ==|	logical double barline, visually rendered as single normal line
  -  |	        not a barline
  -  29|	not a barline

## Essen Folksongs Data

In [ ]:
ESSEN_ROOT = pathlib.Path("../data/essen")

ESSEN_REGIONS = ["europa", "asia", "africa", "america"]

## Parsing decisions and functions
<<sec:parsing>>
According to the =**kern= specification, each token consists of a combination of signifiers
encoding duration, pitch, accidentals, rests, ornaments, articulation, beaming, editorial
marks, and other information.

For the purpose of melodic expectation modeling, we define *musical events* as
notes and rests with the following properties:

- *Kept information*:
  - Duration digits and dots (=0–9= and =.=).
  - Pitch letters (=a–g= / =A–G=) and accidentals (_ =, =-=, =#=, =n=).
  - Rest marker =r= (and potentially pause =;=) to represent silence.

- *Removed information*:
  - Ornaments (trills, mordents, turns, etc.).
  - Appoggiaturas, acciaccaturas (grace notes), and groupettos.
  - Articulation, bowing, stem direction, beaming, partial beaming.
  - User-defined and editorial marks.

This follows the manual’s distinction between core syntactic information and
decorative or editorial notation and matches common practice in IDyOM-style
melodic modeling where durationless ornaments and purely notational details
are excluded from the event stream.

#TODO:
GROUPING_CHARS: 

### Elements definitions

In [ ]:
# Core signifier sets
DURATION_CHARS = set("0123456789.")
PITCH_CHARS = set("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ")
ACCIDENTAL_CHARS = set("_-#n")
REST_CHAR = "r"
PAUSE_CHAR = ";"
GROUPING_CHARS = set("{}()[]")

# Non-core signifiers (to be removed)
ORNAMENT_CHARS = set("Mm$STtwWRO")     # ornaments
APPOGGIATURA_CHARS = set("pP")          # appoggiaturas
GRACE_CHAR = "q"                        # acciaccaturas
GROUPETTO_CHAR = "Q"                    # groupettos
ARTICULATION_CHARS = set("z'\"`^~:I")   # articulations
BOWING_CHARS = set("uv")                # bowing positions
STEM_DIR_CHARS = set("/\\")             # stem direction
BEAM_CHARS = set("LJ")                  # beaming
PARTIAL_BEAM_CHARS = set("kK")          # partial beaming
EDITORIAL_CHARS = set("xXyY?")          # editorial marks
USER_MARK_CHARS = set("iltNUVZ@%+")     # user-defined marks

NON_CORE_CHARS = (
    ORNAMENT_CHARS
    | APPOGGIATURA_CHARS
    | ARTICULATION_CHARS
    | BOWING_CHARS
    | STEM_DIR_CHARS
    | BEAM_CHARS
    | PARTIAL_BEAM_CHARS
    | EDITORIAL_CHARS
    | USER_MARK_CHARS
)

### Helper functions
<<sec:helper>>
Above we defined which elements and characters are core for the analysis and processing
of data in order to prepare it for modelling. In this section we will define helper functions
to recognise said characters and clean the =**kern= files to prepare them for EDA and modelling.

Instead of directly defining a helper function for each unnecessary element in the data, we
just focus for now on null tokens and durationless ornaments that were defined previously to
avoid overcrowding later on the full cleaning function. Elements such as interpretations,
tandems and comments are easily included in the bigger function, having small helper functions
would be a good addition but it is not necessary for such small elements.

Since these do not represent musical events, we do not need to include them in the process.
Better skip them now to avoid unnecessary elements in the data.

Even though for a more detailed and elaborate study of the musical data elements such as
acciaturas and groupettos might be useful and interesting, we will opt to remove them for now.
In case we want to work with their inclusion later on, we can simply rerun the cleaning of
the data without the removal of these characters.

In [ ]:
def is_null_token(line):
  return line.startswith(".")

def has_grace_or_groupetto(tok: str) -> bool:
  # Durationless ornaments that should not be counted as events.
  return (GRACE_CHAR in tok) or (GROUPETTO_CHAR in tok)

We will stript all ornaments, bowing, stems, beamings and editorial marks as well to leave only
the minimal syntactic information needed to define duration and pitch (including rests).
This way we focus on structural melodic content rather than detailed notations.

In [ ]:
def strip_non_core_signifiers(tok: str) -> str:
    """
    Remove ornaments, articulation, beams, editorial marks and other
    non-core signifiers, preserving duration, pitch, accidentals and rest.
    """
    return "".join(ch for ch in tok if ch not in NON_CORE_CHARS)

### Parsing tokens

The =parse_kern_token= function is the main function for the parsing process.
It gathers the previous helper functions defined above in order to create a single
=**kern= token into KernEvents.
Why do we create a class for this?
- We want to turn messy tokens into a structured, clean and homogenous event that will make it easier
  to analyse and feed the models later on.

In this section, we focus on the cleaning of the data. Ornaments and editorial notations are not
necesary for our purpose and will just end up creating noise for the ultimate modelling section.

In [ ]:
@dataclass
class KernEvent:
    duration: str
    pitch: str
    is_rest: bool


def parse_kern_token(tok: str) -> Optional[KernEvent]:
    tok = tok.strip()
    if not tok or is_null_token(tok):
        return None
    if has_grace_or_groupetto(tok):
        return None

    tok = strip_non_core_signifiers(tok)

    # strip grouping / phrasing markers that can wrap tokens
    while tok and tok[0] in GROUPING_CHARS:
        tok = tok[1:]
    while tok and tok[-1] in GROUPING_CHARS:
        tok = tok[:-1]
    if not tok or is_null_token(tok):
        return None

    # Extract duration (leading digits + dots)
    i = 0
    dur_chars = []
    while i < len(tok) and tok[i].isdigit():
        dur_chars.append(tok[i])
        i += 1
    while i < len(tok) and tok[i] == ".":
        dur_chars.append(tok[i])
        i += 1
    duration = "".join(dur_chars)
    if not duration:
        return None

    pitch_part = tok[i:]
    if not pitch_part:
        return None

    is_rest = (REST_CHAR in pitch_part) or (PAUSE_CHAR in pitch_part)
    if is_rest:
        pitch_norm = "r"
    else:
        pitch_chars = [ch for ch in pitch_part if ch in PITCH_CHARS or ch in ACCIDENTAL_CHARS]
        if not pitch_chars:
            return None
        pitch_norm = "".join(pitch_chars)

    return KernEvent(duration=duration, pitch=pitch_norm, is_rest=is_rest)

#### Example

In [ ]:
# Some example **kern note tokens, including ornaments and rests.
tokens = [
    "4c",          # quarter-note C
    "8g#",         # eighth-note G sharp
    "4cL",         # quarter-note C with beaming (L) that should be stripped
    "16bq",        # grace-note B (q) that should be discarded entirely
    "8.cc#",       # dotted eighth-note C-sharp
    "4r",          # quarter rest
    "4G$'",        # quarter-note G with ornament ($) and articulation (') to drop
    ".",
]

for tok in tokens:
    ev = parse_kern_token(tok)
    print(f"token={tok!r} ->", ev)

## Loading and cleaning the Essen corpus
This section walks over the Essen folder structure (grouped by country/region),
parses each file using the token‑level rules above.
The result is a list of cleaned event sequences with metadata.

We loop over each region found in the Essen data folder. For each folder we recursively
find all Humdrum =**kern= files.
This function was mainly created to check whether all the files were available and selectable for
for the next steps of the project. Based on our results, the files are readable and can be used
for further processing.

In [ ]:
def iter_essen_files():
    """
    Yield (path,) for every .krn file anywhere under data/essen.
    """
    for path in ESSEN_ROOT.rglob("*.krn"):
        yield path

In [ ]:
files_seen = list(iter_essen_files())
print("Files seen:", len(files_seen))

In [ ]:
for i, p in enumerate(iter_essen_files()):
    print(i, "->", p)
    if i == 9:
        break

After obtaining the =**kern= files, we need to extract from the full file the musical
representations that we need for future training as explained in the [[sec:parsing][parsing section]]
and the [[sec:helper][helper functions]] definition.
This way, we apply the classification rules from the Humdrum spec:
We skip empty lines and the following attributes:
- global and local comments
- interpretation lines that declare spyne types
- tandem interpretations (e.g., name of voices)
- barlines

With this function we apply the cleaning previously defined with =parse_kern_tokens= on
the full Humdrum file. 

For the remaining lines, we split the data on tab and take the first token, assuming monophonic
melodies and one relevant spine per file (which is how Essen is structured for melodic analysis)

In [ ]:
def load_kern_melody(path: pathlib.Path):
    out = []
    raw_tokens = 0
    parsed_tokens = 0

    with path.open("r", encoding="utf-8", errors="ignore") as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue

            if line.startswith(("*", "!", "=", ".")):
                continue

            tok = line.split("\t")[0].strip()
            raw_tokens += 1

            ev = parse_kern_token(tok)
            if ev is not None:
                out.append(ev)
                parsed_tokens += 1

    return out

test_path = pathlib.Path("data/essen/america/mexico/mexico02.krn")
events = load_kern_melody(test_path)
print("EVENTS:", len(events))
for i, ev in enumerate(events[:10]):
    print(i, ev)

Now, as the last step, we build the full clean corpus.
This function iterates over each file (as shown before all files are captured).
For each file, we obtain its events with the previously defined =load_kern_melody= function.
After obtaining the files, we derive a simple contry label from the path by taking the second
element of the relative path. This way we can group the results based on the countries and regions.

The function returns two lists, the =melodies= with the events (KernEvent) from each melody.
And second, the =meta_list= contains the metadata from each file, that is its name, path, region,
country and length of the melody.

In [ ]:
MIN_EVENTS = 20

def collect_clean_essen():
    melodies = []
    meta_list = []

    for path in ESSEN_ROOT.rglob("*.krn"):
        events = load_kern_melody(path)
        # if len(events) < MIN_EVENTS:
        #     continue

        # derive continent/country from relative path
        rel_parts = path.relative_to(ESSEN_ROOT).parts
        continent = rel_parts[0] if len(rel_parts) > 0 else "unknown"
        country = rel_parts[1] if len(rel_parts) > 1 else "unknown"

        melodies.append(events)
        meta_list.append({
            "filename": path.name,
            "path": str(path),
            "continent": continent,
            "country": country,
            "length": len(events),
        })
    return melodies, meta_list

In [ ]:
melodies, meta = collect_clean_essen()
print("Number of melodies:", len(melodies))

In [ ]:
print(len(melodies[0]))

Let's inspect some of the melodies

In [ ]:
idxs = random.sample(range(len(meta)), 20)
for i in idxs:
    print(i, meta[i]["continent"], meta[i]["country"], meta[i]["filename"], "len:", meta[i]["length"])

In [ ]:
m0 = melodies[0]
print("First melody: length", len(m0))
print("First 10 events:")
for i, ev in enumerate(m0[:10]):
    print(i, ev)

In [ ]:
events_df = []
for i, (events, m) in enumerate(zip(melodies, meta)):
    for j, ev in enumerate(events):
        row = m.copy()
        row.update({
            'melody_id': i,
            'position': j,
            'pitch': ev.pitch,
            'duration': ev.duration,
            'is_rest': ev.is_rest
        })
        events_df.append(row)

df_vp = pd.DataFrame(events_df)
print(df_vp.shape)
print(df_vp.sample(5))

In [ ]:
print(df_vp.head(15))

## Duplicates removal
We need to make sure we do not include any duplicate melodies in the corpus, otherwise we would be introducing
a bias.
We can define melodies being duplicates based on two conditions:
1. Whether the melodies share the same notes (pitches) and duration
2. Whether the melodies share the same pitch intervals and duration

With the second case we account as well for transposition duplicates, that is the same melody
in different pitches (for example, different octaves, or a fifth or a third apart, etc.)

In order to get the full pitcture, we will first change the df format so that all the pitches and
durations belonging to each melody are stored together in a list. That is each melody is associated to
a list of pitches and durations. For now we will do the removal based solely on the exact duplicates,
since this is what was approached on previous studies (Pearce, 2018).

Same pitch and duration:

In [ ]:
prep = (df_vp.sort_values(["melody_id", "position"])
             .groupby("melody_id")
        .apply(lambda l: tuple(zip(l["pitch"], l["duration"]))))

df_dupe = pd.DataFrame({"melody_id": prep.index, "signatures": prep.values})

print(df_dupe.head())

In [ ]:

# Removing exact duplicates

melody_signatures = (df_dupe.drop_duplicates("melody_id")
                           .set_index("melody_id")["signatures"])

seen = {}
keep_ids = []
exact_dupes = []

for mel_id, sig in melody_signatures.items():
    sig = tuple(sig)
    if sig in seen:
        exact_dupes.append({"removed_id": mel_id, "duplicate_of": seen[sig]})
    else:
        seen[sig] = mel_id
        keep_ids.append(mel_id)

print("Original length of full dataframe:", len(df_dupe))
print("How many melodies are duplicates of others:", len(exact_dupes))
print("", len(keep_ids))

# pprint.pp(exact_dupes)

unique_melodies = melody_signatures[keep_ids]
print(unique_melodies.sample(10))

In [ ]:
print(len(unique_melodies))

In [ ]:
df_dupes = pd.DataFrame(exact_dupes)
print(df_dupes.head())

We can check now to which melodies each duplicate id corresponds to and drop them to avoid any biases
when training IDyOM and our Transformer.

In [ ]:
df_paths = df_vp[["filename", "path", "melody_id"]].drop_duplicates()

duplicate_paths = df_dupes.merge(df_paths, left_on="removed_id", right_on="melody_id", how="inner")

paths_removed = duplicate_paths["path"].to_list()

In [ ]:
for path in paths_removed:
  os.remove(path)

Let's check whether the amount of files in our corpus has now changed:

In [ ]:
print(len(list(iter_essen_files())))

Which is exactly 8472 - 269, all the duplicate files have been removed. We can now add the final
cleaned corpus to the IDyOM database and start experimenting with it.

# Data Validation with Humdrum Tools
## Selecting data
The Humdrum data type counts as well with its tool kit that allows for the inspection and
cleaning of raw =**kern= files. It is interesting to use it to double check wheter the manual
preprocessing done is correct. Based on the results, the data obtained from the files corresponds
to what is obtained from the Kumdrum files.

This code block does the following:
- rid removes interpretive data, that is, it strips non-musical lines from Humdrum files.
- G removes global comments
- L removes local comments
- I removes interpretations

Next, =awk -F'\t' '{print $1}'= takes care of selecting only the first spine of the kern file,
that is, in case we have any polyphonies on our corpus, it will be turned into a monophony by
solely selecting the first voice (spine). The field separator (F) is set to tabs, since that is
the standard separations on kern files.

Finally, we get rid of the barlines, as explained before, they offer no musical information, they
are just structural separators that do not bring meaninful information for future modelling.

In [ ]:
%%bash
rid -GLI ../data/essen/america/usa/usa02.krn \
  | awk -F'\t' '{print $1}' \
  | grep -v '^=' \
  > /tmp/island02.tokens

head -n 20 /tmp/island02.tokens

## Validation

In [ ]:
%%bash
find ../data/essen -type f \( -name "*.krn" -o -name "*.kern" \) | sort > essen_files.txt
wc -l essen_files.txt

In [ ]:
%%bash
mkdir -p audit/logs

while read -r f; do
  base=$(basename "$f")
  mint "$f" >audit/logs/"$base".out 2>audit/logs/"$base".err
  echo "$?,${f}" >> audit/validator_exitcodes.csv
done < essen_files.txt

In [ ]:
%%bash
rid -GLI data/essen/europa/misc/island01.krn | extract -i '**kern' > out 2> err
status=$?

echo "$status,$f" >> audit/extract_exitcodes.csv

## Viewpoint availability

Similar to what we did in the cleaning of the files with Python, in this case we do it with the Humdrum
tools to double check the cleaning worked well (even though it was shown above that it was). This time
we also take into account detecting notation that might not be found in files but that it is crucially
important for the future creation of "viewpoints". In this step we want to make sure that we only
keep the files that contain all the necessary information, that is for example duration, pitch, keys,

### Pitch
We want to check that all the files include pitch values, otherwise the file would be empty and
would be useless for our model.

In this case, we a file is flagged if all non-bar tokens are rests (r) or empty.

In [ ]:
%%bash
rm -f audit/no_pitched_notes.csv
: > audit/no_pitched_notes.csv

while IFS= read -r f; do
  f="${f%$'\r'}"   # guard against CRLF

  if ! LC_ALL=C extract -i '**kern' "$f" \
    | sed 's/^,//' \
    | grep -Ev '^(!!!|!!|!|\*|=|\.|$)' \
    | grep -Eq '[abcdefgABCDEFG]'; then
    echo "$f" >> audit/no_pitched_notes.csv
  fi
done < essen_files.txt

wc -l audit/no_pitched_notes.csv

In this process, we take into account several details from each file, including some of the files
contained commas after each note, and cleaning for elements that do not express musical information.
We conduct a similar cleaning process as what was done with =load_kern_melody=. However, in this case we
want to make sure that all the files contain actual pitch representations to avoid using invalid melodies.

The final result gives us one single file. Let's inspect it and see what seems to be the issue and whether
it is worthy to keep it or remove it from the corpus.

In [ ]:
%%bash
extract -i '**kern' ../data/essen/asia/china/han/han0953.krn | head -n 30

Nothing seems to be returned, we can take a look into it with =load_kern_melody= and inspect whether the
events are null or actuall appear.

In [ ]:
events = load_kern_melody(pathlib.Path("data/essen/asia/china/han/han0953.krn"))
print("EVENTS:", len(events))

We see that this file is empty, just contains commentsm interpretations and editoral marks,
elements that do not symbolize musical events, and therefore not important for our research.

In [ ]:
os.remove(pathlib.Path("data/essen/asia/china/han/han0953.krn"))

### Duration

In [ ]:
%%bash
rm -f audit/missing_duration.csv
: > audit/missing_duration.csv

while IFS= read -r f; do
  f="${f%$'\r'}"

  if LC_ALL=C extract -i '**kern' "$f" \
    | sed 's/^,//' \
    | grep -Ev '^(!!!|!!|!|\*|=|\.|$)' \
    | sed -E 's/^[\[{(]+//' \
    | sed -E 's/[}\]]+$//' \
    | grep -Ev '^[0-9]' \
    | grep -q .; then
    echo "$f" >> audit/missing_duration.csv
  fi
done < essen_files.txt

wc -l audit/missing_duration.csv

After taking into account all the elements and structures that are do not needed (as
previously explained in the Pitch cleaning section), we can see that all the files contain
duration information, all of them are valid and we can keep them for future modeling and
analysis.

### Structural integrity

In [ ]:
%%bash
> audit/extract_fail.csv

while read -r f; do
  if ! rid -GLI "$f" | extract -i '**kern' > /dev/null 2>&1; then
    echo "$f" >> audit/extract_fail.csv
  fi
done < essen_files.txt

wc -l audit/extract_fail.csv

## Statistical Analysis

Humdrum tools only provide statistial tools to inspect each kern file individually. If we want
to inspect all the songs at the same time, one option is to concatenate all the songs together,
but this could in turn provide misleading information regarding the overall statistical data
obtained.
Nevertheless, it is interesting to look into how these tools perform and what their output tell us
about our data.

### Intervals
The function =mint= looks into the melodic intervals of each song

In [ ]:
%%bash
mint ../data/essen/america/mexico/mexico02.krn | grep -v '^[!*]' | head -n 10

We can obtain a bit more of information on it with the count of the occurrance of each interval.

In [ ]:
%%bash
mint ../data/essen/america/mexico/mexico02.krn | grep -v '^[!=*]' | sort | uniq -c 

From this analysis we can see that the most common (with the higher number of occurrences) intervals are
P1 (perfect unison) and -M2 (descending Major Second). According to the paper "Predicting Variation of Folk
Songs" (Janssen et al., 2017), the most common and influential melodic intervals are small pitch intervals,
a standard feature of the Dutch folk song corpus.
The presence and frequency of these small intervals carry specific psychological and structural meanings in the
context of folk song variation:

• **Predictor of Stability**: Phrases containing small pitch intervals are more stable and resist change more effectively
as they are transmitted through oral tradition. In other words, phrases with smaller intervals are less subject to
variation over time than those with large leaps

• **Facilitation of Recall**: Small intervals enhance the memorability of a melody. Because these intervals are
easier for singers to remember and reproduce, they contribute to the long-term retention of the piece in a musical tradition.

• **Melodic Expectancy (Pitch Proximity)**: Small intervals correspond to the principle of pitch proximity, which
states that listeners inherently expect the next note in a melody to be close in pitch to the one they just heard. The
paper uses "high average proximity" as a formalization of expectancy—meaning phrases with small intervals contain
"highly expected melodic material".

In summary, the prevalence of small intervals in folk songs means these pieces are structured for efficient communication
and recall, aligning with human cognitive predispositions that favor predictable, proximate pitch patterns.

We can obtain the occurance of these type of intervals on the whole corpus with a bit more work and check whether these
statements are also present in the corpus in general. In the following code block we read all the files, instead of
concatenating them all together, =mint= is called individually for each melody inside the while loop:

In [ ]:
%%bash
while IFS= read -r f; do
  f="${f%$'\r'}"
  continent=$(echo "$f" | awk -F'/' '{print $4}')
  mint "$f" 2>/dev/null \
    | rid -GLId \
    | sed 's/[{}()]//g' \
    | grep -Ev '^(r|$)' \
    | sed "s/^/${continent},/"
done < essen_files.txt > ../audit/mint_by_continent.csv

In [ ]:
df_int = pd.read_csv(
    "../audit/mint_by_continent.csv",
    header=None,
    names=["continent", "interval"]
)

mask = df_int['interval'].astype('str').str.contains('=')
df_int = df_int[~mask]
print(df_int)

Throughout the corpus it still seems that small pitch intervals are the most occurrent in this corpus.
This translates to a similar behaviour of folk music patterns across countries and regions.
We can count the occurances of each interval per continent and check how this shows in a graph for more visual aid:

In [ ]:
print(df_int.groupby(["continent"])["interval"].value_counts().groupby("continent").head(5).to_frame())

This shows the top 5 more repeated intervals per continent. As expected, the most common are very small intervals,
such as minor and major seconds, as well as perfect unison. We cannot confirm this is the general rule for africa
since we only count with one melody belonging to this continent, however it still fits the assumptions and the
conclusions drawn from the paper "Predicting Variation of Folk Songs" (Janssen et al., 2017).

In [ ]:
df_top = (
    df_int
    .groupby("continent")["interval"]
    .value_counts()
    .groupby("continent")
    .head(10)
    .rename("count")
    .reset_index()
)

# Normalize per continent
df_top["proportion"] = df_top.groupby("continent")["count"].transform(
    lambda x: x / x.sum()
)

plt.figure(figsize=(14, 5))
g = sns.barplot(
    data=df_top,
    x="interval",
    y="proportion",
    hue="continent",
    order=df_int["interval"].value_counts().head(10).index,
    palette="viridis"
)

sns.move_legend(g, "upper right", title="Continents")

plt.xticks(rotation=45)
plt.title("Top intervals by continent (normalized)")
plt.tight_layout()
plt.savefig("interval_by_continent.png", dpi=80, bbox_inches="tight")
plt.close()

[[file:interval_by_continent.png]]

From the graph we can observe that even though each continent presents small intervals as most employed in their
melodies, there are still some differences among the use of intervals and their usage.
For example, each continent uses in a similar proportion the descending M2 interval (major second).
While America, together with Europa, presents the higher proportion of P1 (perfect unison), Asia and Africa show
a smaller yet similar proportion in the usage of this interval. Interestingly, Africa shows the highest proportion
of minor intervals, ascending and descending. We cannot take this as the ground truth for this continent since we
only count with one song belonging to this region.
Overall, we see a very similar representation of intervals across the four continents represented in this corpus.

### Entropy (Information Content)

=infot=: calculate information theory measures for Humdrum inputs.
provides a general-purpose tool for measuring the probability relationships
between user-selected data tokens.

=-b= output information (in bits) for each unique data token. The output left column identifies each unique state
and the right column provides the information content for that state measured in bits. 

The key intuition: the greatest information content (lowest probability) is associated with the rarest tokens,
while the lowest information content (highest probability) is associated with the most common tokens.

In [ ]:
%%bash
infot -b -x ^= ../data/essen/america/mexico/mexico03.krn

In [ ]:
%%bash
while IFS= read -r f; do
  f="${f%$'\r'}"
  result=$(mint "$f" 2>/dev/null \
    | rid -GLId \
    | sed 's/[{}()]//g' \
    | grep -Ev '^(r|$)' \
    | infot -b 2>/dev/null \
    | awk '{sum += $NF; n++} END {if (n > 0) printf "%.4f", sum/n}')

  if [ -n "$result" ]; then
    echo "$f,$result"
  fi
done < essen_files.txt > ../audit/infot_intervals.csv

In [ ]:
%%bash
head -10 ../audit/infot_intervals.csv

# Preparing final data structure for training
## Pitch representation

We will start with focusing on the pitch as the basic viewpoint to take into consideration
since it is one the most representative viewpoints on IDyOM, as well as one of the most
representative attributes of a note.

For that, we would need to change the krn notation into smth that Transformers can manipulate.
The most efficient way is transforming them into integers, that can be the abc notation or the
Midi notation. This would a very easy task since we can use the library =music21=

However, krn notatation uses n when mentioning a natural note after an altered note, something
that =music21= does not understand when parsing the pitch. Another issue are repeated letters,
as previously explained, these symbolise octaves. One way to solve this would be to convert the
kern file into a midi file:

In [ ]:
from music21 import converter

score = converter.parse("../data/essen/europa/misc/island02.krn")

score.write("midi", fp="island02.midi")

But this way gives us a bit of a round path since we would then need to parse the midi file
and then finally extract the pitch numbers. Instead, we can create some rules in order for
music21's Pitch function can properly recognise the pitch values in kern and correctly parse
them into midi values (integers).

Rests are also not included in midi notation, so for now we can keep only the notes that are
not a rest. Luckily, we already accounted for this when loading and cleaning the Essen corpus,
therefore we just need to keep those rows that are not a rest, that is that contain a False
in the "is_rest" column.

There are also other edge cases that music21 does not recognise in the kern notation, so we
can do a manual mapping to avoid having any missing notes or errors when parsing the corpus.

In [ ]:
def kern_to_midi(k_pitch):
    note_map = {'c': 0, 'd': 2, 'e': 4, 'f': 5, 'g': 7, 'a': 9, 'b': 11}
    
    letter = k_pitch[0].lower()
    
    if k_pitch[0].islower():
        # count consecutive repetitions of the letter
        count = 0
        for ch in k_pitch:
            if ch.lower() == letter:
                count += 1
            else:
                break
        # c=4 and cc=5
        octave = 3 + count
    else:
        count = 0
        for ch in k_pitch:
            if ch.upper() == k_pitch[0]:
                count += 1
            else:
                break
        # C=3 and CC=2
        octave = 4 - count
    
    midi = 12 * (octave + 1) + note_map[letter]
    
    # accidentals after the letters
    remainder = k_pitch[count:]
    midi += remainder.count('#')
    midi -= remainder.count('-')
    # 'n' = natural, no change
    
    return midi

In [ ]:
df_notes = df_vp[df_vp["is_rest"] == False].drop(["is_rest"], axis=1).copy()
df_notes["midi_pitch"] = df_notes["pitch"].apply(kern_to_midi)

In [ ]:
print(df_notes.sample(10))

Since the manual mapping may not be the most accurate and reliable, we can use IDyOM's built in functionality to
export any dataset or database in the desired format, in this case MIDI. But, rather than just parsing
the full corpus to MIDI and parsing everything again, we can easily export the melodies to a lisp file,
where the CPITCH viewpoint is already found in the desired format (MIDI) notation, and compare it to the
obtained pitch. Let's see how this behaves first with a smaller subset, in this case the Shanxi section
of the China dataset (contains only 10 melodies).
The way to obtain the lisp file was a simple IDyOM command: =(db:export-data (db:get-dataset 4) :lisp "./")=

We can try first with a small number of melodies from the xinhua subset in the asia/china folder.
Since it is a small number we can make a quicker comparison.

In [ ]:
def parse_lisp_melodies(filepath, viewpoint=":CPITCH"):
    with open(filepath, 'r') as f:
        text = f.read()

    # Find all melody blocks: ("melody_name" ...)
    melody_pattern = r'\("([^"]+)"\s*\n((?:\s*\((?:\(:[A-Z].*?\)\s*)+\)\s*)+)'
    melodies = {}

    for melody in re.finditer(melody_pattern, text):
        name = melody.group(1)
        # skip dataset name
        if "Dataset" in name or "Corpus" in name:
            continue
        block = melody.group(2)

        # extract corresponding viewpoint from melody
        pitch_pattern = rf'\({viewpoint}\s+(\d+)\)'
        viewpoints = [int(p) for p in re.findall(pitch_pattern, block)]

        if viewpoints:
            melodies[name] = viewpoints
            
    return melodies

shanxi_melodies = parse_lisp_melodies("dataset-4.lisp")
# pprint.pp(df_shanxi_pitch)

for name, pitches in shanxi_melodies.items():
    print(f"{name}: {len(pitches)} notes, first 10: {pitches[:10]}")    

In [ ]:
xinhua = df_notes[df_notes["path"].str.contains("china01")]
print(xinhua.head(10))

They match perfectly so we can assume that both our function was reliable after all.

## Dataframe format
We can now group all the pitches in a melody in the same row for readability for now. We would
need to check whether long format or compressed format are better for our transformer.

In [ ]:
unique_melodies = (df_notes.sort_values(by=["melody_id", "position"])
                   .groupby("melody_id")["midi_pitch"]
                   .apply(list))

unique_melodies = unique_melodies[keep_ids]
unique_melodies = pd.DataFrame({"melody_id": unique_melodies.index, "pitch":unique_melodies.values})
print(unique_melodies.sample(10))
unique_melodies.to_csv("../data/unique_melodies.csv")

Checking into the melodies that were flagged as duplicated

In [ ]:
print(unique_melodies[unique_melodies["melody_id"] == 32])
print(unique_melodies[unique_melodies["melody_id"] == 34])
print(unique_melodies[unique_melodies["melody_id"] == 35])

# EDA
## Melody inspection

Even though on our cleaning and preprocessing steps we immediately applied the selection of only
one voice, it is interesting to check whether any of our songs are actually polyphonic.

Based on the results with python and Humdrum tools, all of our melodies are monophonic, which
makes the processing much easier.

In [ ]:
%%bash
find ../data/essen -name "*.krn" | while read f; do
  grep -m 1 '^\*\*' "$f" | awk -F'\t' '{print NF}'
done | sort | uniq -c

In [ ]:
voice_counts = {}

for f in Path("data/essen").rglob("*.krn"):
    with f.open(encoding="utf-8", errors="ignore") as fh:
        for line in fh:
            if line.startswith("**"):
                n_voices = line.count("\t") + 1
                voice_counts[n_voices] = voice_counts.get(n_voices, 0) + 1
                break

for k in sorted(voice_counts):
    print(f"{k} voices: {voice_counts[k]} files")

Both the Humdrum tools and our manual Python inspection show that we just have monophonic music, which makes
the whole process much easier.
However, this just applies to the Essen corpus, luckly we already accounted for polyphonic music in the preprocessing in case
the other corpus employed (The Meertens Tune Collecion) does indeed contain multiple voices. 

## Obtaining keys

An interesing addition to the data that will be employed later on (as well as for the EDA) is the key
and mode of the melodies.
Humdrum provides tools necessary to estimate both for each melody.

In [ ]:
%%bash
key data/essen/europa/misc/island02.krn

In [ ]:
%%bash
keycor data/essen/europa/misc/island02.krn

This "key" command returns the estimated key of each musical piece using Krumhansl’s tonal hiearchy method.
A constraint is that this command is solely designed to detect major and minor modes (tonal system) (cite). It is unlikely
that the data found in the corpus differs from this system, but it is still important to mention it as a limitation
for more complex musical structures that may be used in the future.
Also, key is unable to distinguish enharmonic spellings involving double- or triple- sharps or flats.
That is, G double-sharp major is identified as A major (cite).

The output shows as well the confidence value associated to the estimated key: this is a relative certainty
measure among all keys it considered, not a probability that the piece “is” A minor in an absolute sense

The third value returned is a correlation coefficient (r) measures how strongly the piece’s pitch-class profile matches an ideal
representation of the estimated key.

In [ ]:
%%bash
mkeyscape -ln data/essen/europa/misc/island01.krn | magick - ./island01_key.png

### Keys from the kern notation

We can as well obtain they key from the kern files notation. The kern notation usually includes
information as tandem interpretations in the very beginning of the file that can be translated
into the key of the melody (without the tonal system).

This information is given by the tonic and the key signature of the melody.
The tonic represents the first note of the scale in which the melody is set (key).
The key signature defines the key of the melody, that is which notes are altered in said
scale. this includes the notes name and whether they contain sharps or flats. The flats are
represented with =#= and the flats with =-=.

The following section focuses on obtaining these elements from each melody.
However, the tonic and key signature do not provide the major and minor keys.
This will be covered in the next section.

In [ ]:
key_tandem_re = re.compile(r'^\*[A-Ga-g](?:#|-)?\s*:\s*$', re.M)
ksig_re = re.compile(r'^\*k\[[^\]]*\]\s*$', re.M)

def key_matches(path: Path=ESSEN_ROOT):
    files = sorted(path.rglob("*.krn"))
    data = []
    for f in files:
        txt = f.read_text(encoding="utf-8", errors="ignore")
        key_m = key_tandem_re.search(txt)
        ksig_m = ksig_re.search(txt)
    
        key_line = key_m.group(0) if key_m else None
        ksig_line = ksig_m.group(0) if ksig_m else None

        data.append({
            'path': str(f),
            'tonic': key_line,
            'key_signature': ksig_line
        })
    return pd.DataFrame(data)


key_df = key_matches(ESSEN_ROOT)

In [ ]:
print(key_df.sample(10))

In [ ]:
%%bash
key data/essen/asia/china/han/han0659.krn

In [ ]:
df = pd.merge(df, key_df, on="path")
print(df.sample(5))

### Estimated keys with Humdrum tools

With kern tools we can easily create a file with the estimated keys.
As shown before, we employ the command =key= to obtain the estimated key of each melody, together with the
correletion coefficient and the confidence value. The following code block reads through each kern file
found in the essen folder and extracts the mentioned key values. Once all the values are obtained,
these are stored into a csv file.

In [ ]:
%%bash
find ../data/essen/ -name "*.krn" | while read f; do
    key "$f" | awk -v file="$f" '
        /Estimated key:/ {
            key=$3" "$4
            r=$5
            conf=$7
            print file "," key "," r "," conf
        }'
done > estimated_keys.csv

After reading through the csv file, we found that there is no clear structure in it. When reading the
csv file, we make sure to take into accont headers and name the columns for easier manipulation.

In [ ]:
df_keys = pd.read_csv(
  "estimated_keys.csv",
  header=None,
  names=["path", "key", "r", "confidence"]
)

df_keys["r"] = (
    df_keys["r"]
    .str.replace(r"[()r=]", "", regex=True)
    .astype(float)
)

df_keys["confidence"] = (
    df_keys["confidence"]
    .str.replace("%", "")
    .astype(float)
)

print(df_keys.columns)
print(df_keys.head())

In [ ]:
df_keys["mode"] = df_keys["key"].str.contains("minor").map(
    {True: "minor", False: "major"}
)

df_keys["tonic"] = (
    df_keys["key"]
    .str.replace(" major", "", regex=False)
    .str.replace(" minor", "", regex=False)
)

df_keys["continent"] = df_keys["path"].apply(lambda p: p.split("/")[3])

print(df_keys.head())

In [ ]:
df_keys.to_csv("estimated_keys.csv", index=False)

In [ ]:
tab = (
    df_keys
    .groupby(["continent", "mode"])
    .size()
    .unstack(fill_value=0)
)

tab_prop = tab.div(tab.sum(axis=1), axis=0)
print(tab_prop)

### Visual representation

In [ ]:
ax = tab_prop.plot(
    kind="bar",
    stacked=True,
    figsize=(7,4),
    colormap="viridis"
)

ax.set_ylabel("Proportion")
ax.set_xlabel("Continent")
ax.set_title("Proportion of major vs minor modes by continent (Essen corpus)")
ax.legend(title="Mode", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig("major_minor_proportion.png", dpi=150, bbox_inches="tight")
plt.close()

[[file:major_minor_proportion.png]]

In [ ]:
df_keys["tonic"].value_counts().head(10).plot(kind="bar")
plt.title("Most common tonal centers")

plt.savefig("tonic.png")
plt.close()

[[file:tonic.png]]

## Graphs

In [ ]:
continent_counts = Counter()

for f in ESSEN_ROOT.rglob("*.krn"):
    continent = f.parts[2] if len(f.parts) > 2 else "unknown"
    continent_counts[continent] += 1

print(continent_counts)

### Melody Length Distribution

In [ ]:
files = sorted(ESSEN_ROOT.rglob("*.krn"))
lengths = [len(load_kern_melody(f)) for f in files]

df = pd.DataFrame({"length": lengths})

plt.figure(figsize=(10,7))
sns.histplot(df["length"], bins=50, kde=True)
plt.xlabel("Number of events per melody")
plt.ylabel("Count")
plt.title("Melody length distribution (Essen corpus)")
plt.tight_layout()

plt.savefig("melody_lengths.png")
plt.close()

[[file:melody_lengths.png]]

### Pitch Frequency

In [ ]:
pitch_counter = Counter()

for f in files:
  for event in load_kern_melody(f):
    if not event.is_rest:
      pitch_counter[event.pitch] += 1

print("Unique pitches:", len(pitch_counter))
pitch_counter.most_common(20)

In [ ]:
# plt.figure(figsize=(10,4))
# labels, values = zip(*pitch_counter.most_common(30))
# plt.bar(labels, values)
# # plt.xticks(rotation=90)
# plt.title("Most frequent pitches")
# plt.tight_layout()

# plt.savefig("pitch_common.png")
# plt.close()

df = (
    pd.DataFrame(pitch_counter.items(), columns=["pitch", "count"])
    .sort_values("count", ascending=False)
    .head(30)
)

plt.figure(figsize=(10,6))
sns.barplot(data=df, x="pitch", y="count", palette="viridis")
plt.xticks(rotation=90)
plt.title("Most frequent pitches (top 30)")
plt.tight_layout()

plt.savefig("pitch_freq.png")
plt.close()

[[file:pitch_freq.png]]

### Rhythmic Duration Frequency

In [ ]:
dur_counter = Counter()

for f in files:
    for ev in load_kern_melody(f):
        dur_counter[ev.duration] += 1

df = (
    pd.DataFrame(dur_counter.items(), columns=["duration", "count"])
      .sort_values("count", ascending=False)
      .head(20)
)

plt.figure(figsize=(12, 5))

sns.barplot(
    data=df,
    x="duration",
    y="count",
    palette="viridis"
)

plt.title("Most frequent rhythmic durations (Essen corpus)")
plt.xlabel("Duration (kern reciprocal notation)")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig("rhythm_freq.png", dpi=50)
plt.close()

[[file:rhythm_freq.png]]

In [ ]:
rows = []

for f in files:
    continent = f.parts[2]
    for ev in load_kern_melody(f):
        rows.append((continent, ev.duration))

df = pd.DataFrame(rows, columns=["continent", "duration"])

plt.figure(figsize=(12,5))
sns.countplot(
    data=df,
    x="duration",
    hue="continent",
    order=df["duration"].value_counts().index[:10],
    palette="viridis"
)

plt.figure(figsize=(8,4), dpi=120)
sns.countplot(
    data=df,
    x="duration",
    hue="continent",
    order=df["duration"].value_counts().index[:10],
    palette="viridis"
)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("rhythm_by_continent.png", dpi=120, bbox_inches="tight")
plt.close()

[[file:rhythm_by_continent.png]]

In [ ]:

# TODO: Change to log

rows = []

for f in files:
    durations = [ev.duration for ev in load_kern_melody(f)]
    total = len(durations)
    for d in durations:
        rows.append((f.parts[2], d, 1/total))

df = pd.DataFrame(rows, columns=["continent", "duration", "weight"])

plt.figure(figsize=(12,5))
sns.barplot(
    data=df,
    x="duration",
    y="weight",
    hue="continent",
    estimator=sum,
    palette="viridis"
)

plt.title("Normalized rhythmic distribution per continent")
plt.tight_layout()
plt.savefig("rhythm_norm_continent.png", dpi=100, bbox_inches="tight")
plt.close()

[[file:rhythm_norm_continent.png]]

### Pitch Intervals per Continent

In [ ]:
df_top = (
    df_int
    .groupby("continent")["interval"]
    .value_counts()
    .groupby("continent")
    .head(10)
    .rename("count")
    .reset_index()
)

# Normalize per continent
df_top["proportion"] = df_top.groupby("continent")["count"].transform(
    lambda x: x / x.sum()
)

plt.figure(figsize=(14, 5))
g = sns.barplot(
    data=df_top,
    x="interval",
    y="proportion",
    hue="continent",
    order=df_int["interval"].value_counts().head(10).index,
    palette="viridis"
)

sns.move_legend(g, "upper right", title="Continents")

plt.xticks(rotation=45)
plt.title("Top intervals by continent (normalized)")
plt.tight_layout()
plt.savefig("interval_by_continent.png", dpi=80, bbox_inches="tight")
plt.close()

[[file:interval_by_continent.png]]

### Pitch Interval Frequency

In [ ]:
def get_intervals(df):
    intervals_data = []
    for mel_id, group in df.sort_values(by=["melody_id", "position"]).groupby("melody_id"):
        pitches = group["midi_pitch"].values
        continent = group["continent"].iloc[0]
        for i in range(1, len(pitches)):
            intervals_data.append({
                "melody_id": mel_id,
                "interval": pitches[i] - pitches[i-1],
                "continent": continent
            })
    return pd.DataFrame(intervals_data)

ids = df_notes[df_notes.melody_id.astype(int).isin(keep_ids)]
# print(ids["melody_id"].value_counts())

df_intervals = get_intervals(ids)
print(df_intervals)

### Interval Distribution

In [ ]:
# Pitch interval distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
clipped = df_intervals["interval"].clip(-15, 15)
ax.hist(clipped, bins=np.arange(-15.5, 16.5, 1), color="#2d5a7b", edgecolor="white", alpha=0.85)
ax.set_xlabel("Pitch interval (semitones)")
ax.set_ylabel("Count")
ax.set_title("Pitch interval distribution (overall)")
ax.axvline(x=0, color="red", linestyle="--", alpha=0.5, label="Unison")
ax.legend()

ax = axes[1]
for continent in ["asia", "europa"]:
    subset = df_intervals[df_intervals["continent"] == continent]["interval"].clip(-15, 15)
    ax.hist(subset, bins=np.arange(-15.5, 16.5, 1), alpha=0.5, label=continent, density=True)
ax.set_xlabel("Pitch interval (semitones)")
ax.set_ylabel("Density")
ax.set_title("Pitch interval distribution by continent")
ax.legend()

plt.tight_layout()
plt.savefig("interval_distribution.png", dpi=100, bbox_inches="tight")
plt.close()

[[file:interval_distribution.png]]

### Bigram Notes Transition Heatmap

In [ ]:
# Bigram transition heatmap (pitch class)
pitch_class_names = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]

transitions = np.zeros((12, 12))
for mel_id, group in ids.sort_values(by=["melody_id", "position"]).groupby("melody_id"):
    pitches = group["midi_pitch"].values
    for i in range(len(pitches) - 1):
        transitions[pitches[i] % 12][pitches[i+1] % 12] += 1

row_sums = transitions.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1
trans_prob = transitions / row_sums

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

sns.heatmap(np.log1p(transitions), xticklabels=pitch_class_names,
            yticklabels=pitch_class_names, cmap="YlOrRd", ax=axes[0],
            cbar_kws={"label": "Count (log scale)"})
axes[0].set_xlabel("Next pitch class")
axes[0].set_ylabel("Current pitch class")
axes[0].set_title("Bigram counts (log scale)")

sns.heatmap(trans_prob, xticklabels=pitch_class_names,
            yticklabels=pitch_class_names, cmap="YlOrRd", ax=axes[1],
            cbar_kws={"label": "P(next | current)"})
axes[1].set_xlabel("Next pitch class")
axes[1].set_ylabel("Current pitch class")
axes[1].set_title("Bigram transition probabilities")

plt.tight_layout()
plt.savefig("bigram_heatmap.png", dpi=100, bbox_inches="tight")
plt.close()

[[file:bigram_heatmap.png]]

### Melody Length Distribution

In [ ]:
# Melody length distribution with max_len cutoff
max_len = 128
lengths = ids.groupby("melody_id")["position"].count()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(lengths, bins=50, color="#2d5a7b", edgecolor="white", alpha=0.85)
axes[0].axvline(x=max_len, color="red", linestyle="--", linewidth=2, label=f"max_len = {max_len}")
axes[0].set_xlabel("Melody length (number of notes)")
axes[0].set_ylabel("Count")
axes[0].set_title("Melody length distribution")
axes[0].legend()

n_truncated = (lengths > max_len).sum()
n_total = len(lengths)
stats_text = (f"Total: {n_total}\nMean: {lengths.mean():.1f}\nMedian: {lengths.median():.1f}\n"
              f"Truncated (>{max_len}): {n_truncated} ({n_truncated/n_total*100:.1f}%)")
axes[0].text(0.95, 0.95, stats_text, transform=axes[0].transAxes,
             va="top", ha="right", fontsize=9,
             bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

sorted_lengths = np.sort(lengths)
cumulative = np.arange(1, len(sorted_lengths) + 1) / len(sorted_lengths)
axes[1].plot(sorted_lengths, cumulative, color="#2d5a7b", linewidth=2)
axes[1].axvline(x=max_len, color="red", linestyle="--", linewidth=2, label=f"max_len = {max_len}")
axes[1].set_xlabel("Melody length (number of notes)")
axes[1].set_ylabel("Cumulative proportion")
axes[1].set_title("Cumulative melody length distribution")
axes[1].legend()

plt.tight_layout()
plt.savefig("melody_length_distribution.png", dpi=100, bbox_inches="tight")
plt.close()

[[file:melody_length_distribution.png]]

### N-gram counts per order(?))